# Sample AI Driver Safety Event Analysis Notebook

The application presented in this notebook will analyze videos taken from a truck dash cam (front, driver, and side) and provide a summary of the videos and a severity of the event based on the video. 

### Pre-requisites
1. Amazon SageMaker Studio Lab account 
2. AWS account 
3. Access to Amazon Nova Pro model
4. Sample video files for the truck front view, driver facing view, and side view

In [ ]:
# install library to process the video frames
!pip install opencv-python-headless

In [ ]:
# setup credentials and region
import os
import boto3

# input variable shown below is using PowerShell syntax; adjust this section based on your needs
#creds = boto3.Session().get_credentials()


In [ ]:
# core functions that will used when executing next cell
import cv2
import numpy as np
import json
import base64
import argparse
import time
from typing import List


# AWS Configuration
AWS_ACCOUNT = boto3.client('sts').get_caller_identity().get('Account')
AWS_REGION = "us-east-1"
inference_profile_arn = f"arn:aws:bedrock:{AWS_REGION}:{AWS_ACCOUNT}:inference-profile/us.amazon.nova-pro-v1:0"

def add_caption(frame, text):
    """
    Add caption to the top of the frame
    
    Args:
        frame (numpy.ndarray): Input frame
        text (str): Caption text
    """
    text_height = 40
    caption_bg = np.zeros((text_height, frame.shape[1], 3), dtype=np.uint8)
    
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    font_thickness = 2
    text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
    
    text_x = (frame.shape[1] - text_size[0]) // 2
    text_y = (text_height + text_size[1]) // 2
    
    cv2.putText(caption_bg, text, (text_x, text_y), font, font_scale, (255, 255, 255), font_thickness)
    
    return np.vstack((caption_bg, frame))


def combine_frames(videos, output_dir='/tmp/combined_frames', target_fps=4):
    """
    Combine frames from multiple videos side by side at specified FPS
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    else:
        # Clean up existing files
        for file in os.listdir(output_dir):
            os.remove(os.path.join(output_dir, file))                    
        
    captions = ["Front View", "Driver View", "Side View"]
    captures = [cv2.VideoCapture(video) for video in videos]   
   
    frame_counts = [int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) for cap in captures]
    print("frame_counts : ",frame_counts)
    fps_rates = [cap.get(cv2.CAP_PROP_FPS) for cap in captures]
    print("fps_rates : ",fps_rates)
    
    skip_frames = [int(fps/target_fps) for fps in fps_rates]
    min_frames = min(frame_counts)
    print("skip_frames : ",skip_frames)
    print("min_frames : ",min_frames)
    print("target_fps : ",target_fps)
    
    
    frame_number = 0
    output_frame_number = 0
    frame_paths = []
    
    while frame_number < min_frames:
        frames = []
        for cap_idx, cap in enumerate(captures):
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_number % skip_frames[cap_idx] == 0:
                frames.append(frame)
            else:
                continue
        
        if len(frames) != len(videos):
            frame_number += 1
            continue
            
        heights = [frame.shape[0] for frame in frames]
        min_height = min(heights)
        
        resized_frames = []
        for idx, frame in enumerate(frames):
            aspect_ratio = frame.shape[1] / frame.shape[0]
            new_width = int(min_height * aspect_ratio)
            resized = cv2.resize(frame, (new_width, min_height))
            resized_with_caption = add_caption(resized, captions[idx])
            resized_frames.append(resized_with_caption)
            
        combined_frame = np.hstack(resized_frames)
        
        output_path = os.path.join(output_dir, f'frame_{output_frame_number:04d}.jpg')
        cv2.imwrite(output_path, combined_frame)
        frame_paths.append(output_path)
        
        output_frame_number += 1
        frame_number += 1
    
    for cap in captures:
        cap.release()
    
    return frame_paths, output_frame_number


def analyze_frames_with_bedrock(frame_paths: List[str], inference_profile_arn: str, batch_size: int = 20) -> str:
    """
    Analyze frames using Amazon Bedrock Converse API
    
    Args:
        frame_paths (List[str]): List of paths to frame images
        inference_profile_arn (str): ARN of the Bedrock inference profile
        batch_size (int): Number of frames to process in each batch
    
    Returns:
        str: Summary of the video analysis
    """
    bedrock_runtime = boto3.client(
        service_name="bedrock-runtime",
        region_name=inference_profile_arn.split(":")[3]  # Extract region from ARN
    )

    all_observations = []
    
    # Process frames in batches
    for i in range(0, len(frame_paths), batch_size):
        batch_paths = frame_paths[i:i + batch_size]
        
        # Prepare the messages for conversation
        messages = []
        for frame_path in batch_paths:
            # Read image as bytes without encoding
            with open(frame_path, "rb") as image_file:
                image_bytes = image_file.read()
                
                messages.append({
                    "role": "user",
                    "content": [{
                        "image": {
                            "format": "jpeg",
                            "source": {
                                "bytes": image_bytes  # Send raw bytes directly
                            }
                        }
                    }]
                })
        
        # Add the analysis request message
        messages.append({
            "role": "user",
            "content": [{
                "text": "Analyze these synchronized camera views and describe what you observe. "
                        "Focus on any safety concerns or notable events."
            }]
        })
        
        try:
            # Use the Converse API
            response = bedrock_runtime.converse(
                modelId=inference_profile_arn,
                messages=messages,
                system=[{
                    "text": "You are an expert at analyzing multi-camera surveillance footage. "
                           "Provide detailed observations about activities, risks, and notable events "
                           "from the synchronized camera views (Front, Driver, and Side views)."
                }]
            )
            
            # Extract content from response
            if 'output' in response and 'message' in response['output'] and 'content' in response['output']['message']:
                observation = response['output']['message']['content'][0]['text']
                all_observations.append(observation)
                print('\n Analysis### :: \n', observation)
            else:
                print(f"Unexpected response format: {response}")

            token_usage = response['usage']
            print(f"\nToken Usage Iteration :", i+1)
            print(f"Input tokens:  {token_usage['inputTokens']}")
            print(f"Output tokens: {token_usage['outputTokens']}")
            print(f"Total tokens: {token_usage['totalTokens']}")
#            print(f"Stop reason: {token_usage['stopReason']}")

            #sleep for 1 sec between iterations, otherwise it may hit per sec token limit.
            time.sleep(1)
        
        except Exception as e:
            print(f"Error processing batch starting at iteration {i}: {str(e)}")
            continue
    
    # Generate final summary using Converse API
    try:
        # Combine all observations into a single text
        combined_observations = "\n\n".join(all_observations)
        
        final_response = bedrock_runtime.converse(
            modelId=inference_profile_arn,
            messages=[{
                "role": "user",
                "content": [{
                    "text": f"Please provide a concise summary of the following observations:\n\n{combined_observations}"
                }]
            }],
            system=[{
                "text": "Provide a JSON object with two fields, DO NOT provide any preamble; first one is a severity based on a risk of a high impact accident. Use high, medium, low for severity. For second field, include a short paragraph summary of the event."
            }]
        )

        # Extract content from final response
        if 'output' in final_response and 'message' in final_response['output'] and 'content' in final_response['output']['message']:
            token_usage = final_response['usage']
            print(f"\nToken Usage for Final Summary :")
            print(f"Input tokens:  {token_usage['inputTokens']}")
            print(f"Output tokens: {token_usage['outputTokens']}")
            print(f"Total tokens: {token_usage['totalTokens']}")
#            print(f"Stop reason: {token_usage['stopReason']}")
            return final_response['output']['message']['content'][0]['text']
        else:
            print(f"Unexpected final response format: {final_response}")
            return "Error: Unexpected response format"
    
    except Exception as e:
        print(f"Error generating final summary: {str(e)}")
        return "Error generating summary"

- Collect your videos, upload them to this folder and update the video_files array with the names in this cell
- Make sure you have access to Bedrock Nova Pro model, get the inference profile and update the inference_profile_arn variable in this cell 

In [ ]:
# Process the videos, create combined frames, and analyze them with Bedrock Amazon Nova Pro to get summary

# If you see an error with Converse API token expired but you already confirmed you have right tokens, restart the kernal and run again

# List of input video files
video_files = [
    './app/samples/front.mp4',   # Front view
    './app/samples/driver.mp4',  # Driver view
    './app/samples/side.mp4'     # Side view
]

# Combine frames at specified FPS
frame_paths, total_frames = combine_frames(
    video_files, 
    output_dir='./output_frames', 
    target_fps=2
)
print(f"Successfully processed {total_frames} frames")

# Analyze frames with Bedrock Nova Pro
print("Analyzing frames with Amazon Bedrock Nova Pro...")
summary = analyze_frames_with_bedrock(
    frame_paths,
    inference_profile_arn
)

print("\nAnalysis Summary:")
print(summary)